# report05 — 파형 검증 — **Sionna 로 대조**

> ### ❓ 이 리포트가 답하는 질문
> **우리가 규격서 보고 손으로 만든 통신 신호가, 진짜 규격과 똑같은가?**

### ⚡ 결론부터 (TL;DR)

1. **앞 리포트에서 우리는 통신 신호(WiFi·LTE·5G)를 규격서만 보고 손으로 만들었다.** 손으로 옮긴 것에는 실수가 숨을 수 있다. 그래서 **믿을 수 있는 다른 방법으로 같은 신호를 다시 만들어 겹쳐본다** — 검증된 라이브러리 **Sionna PHY** 의 OFDM 변조기로.
2. **세 신호 모두 사실상 완벽히 일치한다.** 우리 파형과 Sionna 파형의 상관(두 신호가 얼마나 똑같은가, 1이면 완전히 같음)이 WiFi·LTE·5G 모두 **1.0000**. 남은 차이는 **NMSE -135.2 dB ~ -138.3 dB** 로, 이건 물리적 차이가 아니라 컴퓨터가 소수를 저장할 때 마지막 자리에서 반올림하는 정도(float32 한계)일 뿐이다.
3. **이 대조는 규격의 아주 미세한 부분까지 확인해준다.** 예를 들어 5G·LTE 는 슬롯의 **첫 심볼에만 순환전치(CP)를 더 길게** 준다 (LTE [160, 144, 144, 144, …] → 첫 칸 160 vs 이후 144 샘플). 이 한 가지 규칙만 놓쳐도 상관이 1.0000 에서 **0.05** 로 무너진다 — 대조가 그만큼 예민하다는 뜻이다.
4. **Sionna PHY 는 여기서 '검증자'다.** Sionna 에는 WiFi·LTE·SSB 신호 **생성기가 없다**(5G NR 위주). 그래서 신호를 만드는 일은 우리 `waveforms.py` 가 계속 맡고, Sionna 는 **독립적인 OFDM 엔진**으로 같은 격자를 다시 변조해 우리 결과를 **채점**한다.

### 🗺️ 어디부터 읽나

| 절 | 무엇을 |  |
|---|---|---|
| §1 | 왜 검증이 필요한가 — **손으로 옮긴 악보** | 규격서를 손으로 옮기면 실수가 숨는다 |
| §2 | 어떻게 대조하나 — **같은 격자, 두 변조기** | Sionna 는 생성기가 아니라 독립 채점자 |
| §3 | 결과 — **상관 1.0000, 차이는 반올림뿐** | 세 신호 모두 사실상 동일 |
| §4 | 대조가 잡아내는 것 — **첫 심볼의 긴 CP** | 규격의 미세한 조항 하나까지 검사 |
| 바쁘면 | §3 의 결과표와 교차검증 그림만 | 이 리포트의 결론 |

---


## 🔰 5분이면 이해하는 이 리포트

*(수식·표가 부담스러우면 여기만 읽어도 됩니다. 아래 §들은 같은 얘기를 숫자로 증명합니다.)*

**우리는 앞 리포트에서 통신 신호(WiFi·LTE·5G)를 규격서만 보고 손으로 만들었습니다.** 레이더가 표적을 알아보려면 '내가 쏘거나 빌려 쓰는 신호가 정확히 어떻게 생겼는지' 를 알아야 하는데, 그 신호를 우리가 직접 조립한 겁니다. 문제는 — **손으로 옮긴 것에는 실수가 숨을 수 있다**는 점입니다. 규격서는 두껍고, 칸 하나·꼬리 한 조각을 잘못 놓아도 눈으로는 티가 안 납니다.

**그래서 이렇게 확인합니다: 내가 옮겨 적은 악보를, 프로 연주자에게 그대로 쳐보게 하는 겁니다.** 만약 내 악보가 정확하다면, 프로가 친 소리와 원본 녹음이 **음 하나까지 똑같이** 포개져야 합니다. 여기서 '프로 연주자' 역할이 **Sionna PHY** — 전 세계가 검증한 통신 신호 라이브러리 — 입니다. 우리가 만든 자원격자(신호 설계도)를 Sionna 의 변조기에 그대로 넣어, **우리와 전혀 다른 코드로 같은 신호를 다시 만들게** 하고, 두 결과를 겹쳐봅니다.

**결과: 세 신호 모두 완벽히 포개집니다.** 두 신호가 얼마나 같은지 재는 '상관' 값이 WiFi·LTE·5G 전부 **1.0000**(1이면 완전히 동일). 남은 차이는 -135 dB 수준인데, 이건 진짜 차이가 아니라 **컴퓨터가 소수를 저장하며 마지막 자리에서 반올림하는 정도**에 불과합니다. 즉 우리가 손으로 만든 신호는 라이브러리가 만든 신호와 **사실상 완전히 같습니다.**

**게다가 이 대조는 아주 예민합니다 — 규격의 깨알 같은 조항까지 잡아냅니다.** 예를 들어 5G·LTE 는 한 묶음(슬롯)의 **첫 심볼에만 완충 구간(순환전치)을 조금 더 길게** 주라고 정해 놨습니다. 이 사소한 규칙 하나만 놓쳐도, 완벽하던 상관이 1.0000 에서 **0.05 로 무너집니다.** 그러니 상관이 1.0000 이라는 건 그런 미세한 부분까지 전부 맞았다는 뜻입니다.

**한 가지 짚을 점 — Sionna 는 '검증자' 이지 '대체품' 이 아닙니다.** Sionna 에는 WiFi·LTE·5G-SSB 신호를 **처음부터 만들어 주는 기능이 없습니다**(5G 데이터 채널 위주). 그래서 신호를 **만드는 일은 우리 코드가 계속** 맡고, Sionna 는 옆에서 **독립적으로 채점**하는 역할만 합니다. 채점자가 우리와 같은 답을 냈으니, 우리 답이 옳다는 강한 증거가 되는 겁니다.

> **한 줄 요약** — 규격서 보고 손으로 만든 신호가 맞는지, 검증된 라이브러리로 같은 신호를 다시 만들어 겹쳐봤습니다. **세 신호 모두 상관 1.0000** — 사실상 완전히 같고, 남은 차이는 반올림뿐입니다.

## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| 자작 WiFi/LTE/5G 파형 (검증 대상) | 3GPP TS 36.211 · TS 38.211 · IEEE 802.11ac 를 읽어 `src/waveforms.py` 에 구현 | 🔴 우리 구현 (검증 대상) |
| 교차대조용 독립 OFDM 변조기 | **`sionna.phy.ofdm.OFDMModulator`** — 같은 자원격자를 우리와 무관하게 다시 변조 | 🟢 라이브러리 (채점자) |
| 5G NR 뉴머롤로지 (μ·SCS·슬롯·CP 길이) | **`sionna.phy.nr.CarrierConfig`** — 3GPP 표를 Sionna 에게 물어봤다 | 🟢 라이브러리 (3GPP TS 38.211 구현) |
| 상관·NMSE 대조 숫자 | `src/waveforms_sionna.py` 가 두 파형을 겹쳐 계산 → JSON 에 기록 | 📐 측정 결과 |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `sionna-phy` | Sionna PHY (`ofdm`/`nr`/`channel`) — OFDM 변복조 · 3GPP 뉴머롤로지 · RT 경로를 신호에 적용 | 🟢 **Sionna 내부** (PyTorch 백엔드, GPU) |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 짠 **SBR** 이 합니다 — Sionna 에 RCS 솔버가 없기 때문입니다. 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진을 그대로** 씁니다.
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `sionna` | 2.0.1 | 광선추적(RT) + PHY(OFDM/NR/채널) — **이 프로젝트의 중심** |
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `scipy` | 1.18.0 | 스플라인(단면 보간·암 경로) · STFT(스펙트로그램) |
| `matplotlib` | 3.11.0 | 도표 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: 이 리포트는 **재측정을 하지 않는다** — report2 파이프라인이 남긴 JSON·그림을 재배치한다. 노트북 생성은 초 단위. (원본 교차검증 `waveforms_sionna.py` 자체는 GPU 1장·수 초.)

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
cd /home/yunjung/workspace/sionna2

# 교차검증 자체를 다시 돌려 상관·NMSE 를 재계산하려면:
~/.venvs/py312/bin/python src/waveforms_sionna.py   # 자작 <-> Sionna 대조

# 이 리포트의 숫자·그림은 report2 파이프라인이 남긴 JSON·그림을 재사용한다(재측정 없음).
~/.venvs/py312/bin/python src/make_notebook05.py    # JSON -> report05.ipynb
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 측정 스크립트가 JSON 을 남기고, 노트북 생성기(`src/make_notebook*.py`)가 그 JSON 을 읽어 본문에 주입합니다. → **그림과 글이 어긋날 수 없습니다.** 숫자가 이상하면 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/report2_waveform_rcs.json` | `crosscheck` 블록 = **이 리포트의 모든 대조 숫자**(상관·NMSE·CP 길이) |
| `outputs/figures/report2_crosscheck.png` | §3·§4 교차검증 — 시간파형·스펙트럼 포갬 + 첫-심볼 CP 민감도 |
| `outputs/figures/report2_sionna_waveforms.png` | §2·§3 Sionna = OFDM 엔진이자 채점자, 잔차 = float32 반올림 |
| `outputs/figures/report2_numerology.png` | §2 5G 뉴머롤로지 — 우리가 안 짜고 CarrierConfig 에서 읽어온 표 |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- **이 대조가 보증하는 것은 'OFDM 변조 수학'과 '5G 뉴머롤로지' 다.** 즉 IFFT·가드·DC널·심볼별 CP 삽입이 규격대로인가, 그리고 격자 치수(RB·심볼수·CP 길이)가 3GPP 표와 맞는가. 이건 두 독립 구현이 소수점 한계까지 일치함으로 강하게 보증된다.
- **파일럿(기준신호) 배치 위치까지 Sionna 가 독립적으로 확인해 주지는 못한다.** Sionna PHY 에는 WiFi VHT-LTF·LTE CRS·5G SSB **생성기가 없기** 때문이다. 그 배치 좌표는 우리가 3GPP/IEEE 스펙을 읽어 넣은 것이고, 근거는 코드 주석과 `docs/` 에 남겼다. 대조는 그 격자를 **신호로 바꾸는 단계**를 검증한다.
- **여기서 검증하는 것은 '규격 일치' 이지 '탐지 성능' 이 아니다.** 이 파형으로 표적을 얼마나 잘 보는가(거리·속도 분해능)는 앞 리포트(→ report04), 표적이 얼마나 밝은가는 → report06 소관이다.

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| [report04](report04.ipynb) — 조명원 · 파형 (WiFi/LTE/5G) | **앞 리포트.** 그 파형이 **무엇을 주는가**(대역·PRF·분해능)를 다뤘다. 여기서는 그 파형이 **정말 규격대로인가**를 검증한다 |
| **report05 (여기)** — 파형 검증 | 자작 파형 ↔ Sionna OFDMModulator 교차대조. 상관 1.0000 = 규격 일치의 강한 증거 |
| [report06](report06.ipynb) — 드론이 레이더에 얼마나 밝은가 (RCS) | **다음 리포트.** 조명(파형)을 확정했으니, 이제 표적 쪽 — 드론의 되비침 밝기(RCS)로 |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **규격 / 표준(spec)** | 통신 신호가 어떤 칸에 무엇을 담아야 하는지 정한 문서(3GPP·IEEE). 우리는 이걸 읽고 신호를 손으로 만들었다 |
| **OFDM** | 부반송파 수백~수천 개에 데이터를 잘게 나눠 싣는 변조 방식. 이 신호를 만드는 핵심 연산이 IFFT(역푸리에변환)와 CP 삽입 |
| **자원격자(resource grid)** | 시간=가로·주파수=세로의 모눈종이. OFDM 신호는 이 칸을 채워 만든다 |
| **순환전치 CP(cyclic prefix)** | 각 OFDM 심볼 **끝부분을 복사해 앞에 덧붙이는 완충 구간**. 다중경로로 신호가 번져도 심볼끼리 안 겹치게 막는 보호띠 |
| **첫 심볼 긴 CP** | 3GPP 는 한 슬롯의 **첫 심볼에만 CP 를 더 길게** 준다(정렬을 맞추려고). LTE·5G 의 미세하지만 반드시 지켜야 하는 규칙 |
| **상관(correlation)** | 두 신호가 얼마나 똑같은가를 재는 값. **1이면 완전히 같음**, 0이면 무관 |
| **NMSE** | 정규화 평균제곱오차. 두 신호 차이의 크기를 dB 로 표시 — **더 음수(작을수록)일수록 완벽히 같다.** -135 dB 는 사실상 0 |
| **float32** | 컴퓨터가 소수를 32비트로 저장하는 방식. 유효숫자 약 7자리에서 반올림한다 → 두 계산이 완벽히 같아도 마지막 자리에 ~-140 dB 수준의 미세한 차이가 남는다 |
| **뉴머롤로지** | SCS(부반송파 간격)·슬롯·심볼수·CP 길이 등 5G 물리계층 격자 파라미터 |
| **Sionna PHY** | 검증된 오픈소스 물리계층 라이브러리. OFDM 변복조·3GPP 뉴머롤로지를 제공. 여기선 우리 파형을 채점하는 **독립 참조**로 쓴다 |
| **OFDMModulator** | Sionna 의 OFDM 변조기 — 자원격자를 받아 IFFT·CP 삽입으로 시간파형을 만든다 |
| **CarrierConfig** | Sionna 가 3GPP 5G NR 뉴머롤로지 표를 담고 있는 객체. 우리가 표를 안 짜도 됨 |

</details>

---


> **앞 리포트**(report04)에서 우리는 패시브 레이더가 빌려 쓰는 **조명원(WiFi·LTE·5G 파형)** 이 **무엇을 주는가** — 거리·속도 분해능을 다뤘습니다. 그 파형은 우리가 규격서를 읽어 손으로 만든 것이었습니다. 이 리포트는 그 다음 질문에 답합니다: **그 파형이 정말 규격대로인가?**

---
## §1. 왜 검증이 필요한가 — 손으로 옮긴 악보

> 🔍 **여기서 하는 일** — 우리가 신호를 어떻게 만들었고, 왜 그것만으론 못 믿는지 봅니다.

레이더가 되돌아온 신호에서 표적을 찾으려면, **원래 나간 신호가 정확히 어떻게 생겼는지**를 알아야 합니다(그래야 '되돌아온 것' 과 '나간 것' 을 상관으로 맞춰볼 수 있으니까요). 그래서 우리는 통신 신호 세 종류를 **규격서를 읽어 손으로 조립**했습니다:

- **WiFi 802.11ac** — IEEE 802.11ac 규격
- **LTE Rel-9** — 3GPP TS 36.211
- **5G NR Rel-16** — 3GPP TS 38.211

각 신호는 **자원격자(시간=가로·주파수=세로의 모눈종이)** 의 칸마다 무엇을 켤지 규격이 정해두고 있습니다. 그 칸을 우리 손으로 채운 뒤, **OFDM(부반송파 수천 개에 데이터를 잘게 나눠 싣는 변조 방식)** 로 시간축 파형을 만듭니다.

**문제는, 손으로 옮긴 것에는 실수가 숨는다는 점입니다.** 규격서는 두껍고, 부반송파 하나를 잘못 배치하거나, 심볼 앞에 붙이는 **순환전치(CP — 심볼 끝을 복사해 앞에 덧붙이는 완충 구간)** 의 길이를 헷갈려도, 만든 파형만 눈으로 봐서는 티가 나지 않습니다. 스펙트럼도 그럴듯하게 나오죠.

> **비유 —** 두꺼운 악보를 손으로 베껴 적었다고 합시다. 내 사본이 원본과 음 하나까지 같은지, **그냥 종이만 들여다봐선 알 수 없습니다.** 확인하려면 **누군가 실제로 연주**해 원본과 들어맞는지 들어봐야 합니다.

그 '연주' 를 해줄 사람이 필요합니다 — 우리와 **무관하게, 검증된 방법으로 같은 신호를 다시 만들어 줄** 독립적인 도구. 그게 다음 절의 Sionna PHY 입니다.

---
## §2. 어떻게 대조하나 — 같은 격자, 두 변조기

> 🔍 **여기서 하는 일** — Sionna 를 '독립 채점자' 로 세우는 방법과, Sionna 가 무엇을 확인해 주고 무엇은 못 하는지 정직하게 정리합니다.

**Sionna PHY** 는 전 세계가 쓰는 검증된 통신 물리계층 라이브러리입니다. 우리는 이걸 이렇게 씁니다:

1. **같은 자원격자를 준비한다.** 우리 `waveforms.py` 가 스펙대로 채운 바로 그 격자.
2. **두 개의 서로 다른 OFDM 변조기에 똑같이 넣는다.**
   - 하나는 **우리 코드**(`src/waveforms.py` 의 자작 변조기)
   - 하나는 **Sionna** (`sionna.phy.ofdm.OFDMModulator`) — 우리 코드와 한 줄도 안 겹침
3. **두 시간파형을 겹쳐본다.** 같은 격자를 정직하게 변조했다면 결과는 같아야 한다.

두 변조기가 **독립적으로 같은 답**을 내면, 우리 구현이 옳다는 강한 증거가 됩니다(둘이 똑같은 실수를 우연히 저지를 확률은 사실상 0이니까요).

### 뉴머롤로지도 우리가 안 짠다 — Sionna 에게 물어본다

5G 는 격자의 치수(부반송파 간격·슬롯당 심볼수·CP 길이)를 **뉴머롤로지** 라는 규격 표로 정해둡니다. 이 표조차 우리가 손으로 옮기지 않고, Sionna 의 `CarrierConfig` 에서 **읽어옵니다** — '라이브러리가 아는 걸 다시 짜지 말자' 는 원칙입니다.

![numerology](outputs/figures/report2_numerology.png)

*(위 표의 μ·심볼수·슬롯수·CP 유형은 전부 `CarrierConfig(subcarrier_spacing, n_size_grid)` 에서 그대로 읽어온 값입니다. 우리가 유지하는 표가 아닙니다.)*

### 정직하게 — Sionna 는 '검증자' 이지 '대체품' 이 아니다

> ⚠️ **Sionna PHY 에는 WiFi·LTE·5G-SSB 신호 생성기가 없습니다**(5G 데이터 채널 위주). 그래서 신호를 **만드는 일은 우리 `waveforms.py` 가 계속** 맡습니다. Sionna 가 해주는 건 **(1) OFDM 변조를 독립적으로 다시 계산**하고 **(2) 5G 뉴머롤로지를 제공**하는 것 — 즉 **채점**입니다.

그래서 이 대조가 **보증하는 것**과 **못 하는 것**을 분명히 해둡니다:

| 이 대조가 **확인해 주는 것** | 이 대조가 **못 하는 것** |
|---|---|
| OFDM 변조 수학(IFFT·가드·DC널·심볼별 CP)이 규격대로인가 | 파일럿(CRS/SSB/VHT-LTF) **위치**의 독립 확인 |
| 5G 뉴머롤로지(RB·심볼수·CP 길이)가 3GPP 표와 맞는가 | (Sionna 에 그 생성기가 없어 위치는 우리가 스펙대로 넣음) |

즉 대조는 **격자를 신호로 바꾸는 단계**를 소수점 한계까지 검증합니다. 파일럿을 어느 칸에 놓느냐는 우리가 규격을 읽어 넣은 것이고, 근거는 코드 주석과 `docs/` 에 남겼습니다.

---
## §3. 결과 — 상관 1.0000, 남은 차이는 반올림뿐

> 🔍 **여기서 하는 일** — 세 신호를 겹친 실제 숫자를 봅니다.

같은 격자를 두 변조기로 만든 뒤 겹쳤습니다. **세 신호 모두 사실상 완벽히 일치**합니다:

| 표준 | 표본 수 | 표본율 $f_s$ | **상관** (1=동일) | **NMSE** (작을수록 동일) |
|---|---|---|---|---|
| WiFi 802.11ac | 4,160 | 80.00 MHz | **1.0000** | **-138.3 dB** |
| LTE Rel-9 | 30,720 | 30.72 MHz | **1.0000** | **-135.6 dB** |
| 5G NR Rel-16 | 61,440 | 122.88 MHz | **1.0000** | **-135.2 dB** |

**읽는 법 — 상관.** 두 신호가 얼마나 같은가를 재는 값입니다. 1이면 완전히 같고, 0이면 전혀 다릅니다. 세 신호 모두 소수 넷째 자리까지 **1.0000** — 두 신호를 겹쳐 그리면 한 줄로 포개집니다.

**읽는 법 — NMSE.** 두 신호의 '차이의 크기' 를 dB 로 나타낸 값으로, **더 음수일수록(작을수록) 완벽히 같다**는 뜻입니다. 여기서 -135.2 ~ -138.3 dB 는 상상하기 어려울 만큼 작습니다 — 신호 세기 대비 차이가 약 $10^{-13}$ 배라는 뜻이니까요.

### 그럼 그 -135 dB 마저 왜 0이 아닌가? — float32 반올림

**남은 미세한 차이는 물리가 아니라 컴퓨터 산수의 한계입니다.** 컴퓨터는 소수를 **float32**(32비트) 로 저장하는데, 유효숫자 약 7자리에서 반올림합니다. 그래서 두 계산이 수학적으로 완전히 같아도, 서로 다른 순서로 더하고 곱하면 **마지막 자리에 ~-140 dB 수준의 티끌**이 남습니다. 우리가 본 -135 dB 는 바로 그 **반올림 바닥(floor)** 에 붙어 있습니다.

> **비유 —** 같은 계산을 계산기 두 대로 하면 화면엔 같은 답이 뜨지만, 내부적으로 마지막 자릿수만 아주 미세하게 다를 수 있습니다. 그 차이는 '두 계산이 틀렸다' 가 아니라 '계산기가 소수를 딱 그만큼만 정밀하게 다룬다' 는 뜻입니다.

아래 그림이 이걸 한눈에 보여줍니다. 시간파형(위)·스펙트럼(가운데)이 두 색으로 완전히 포개지고, 잔차(오른쪽 아래 패널)는 -135 dB 언저리의 평평한 잡음 — 즉 **float32 반올림** 입니다.

![sionna waveforms](outputs/figures/report2_sionna_waveforms.png)

![.](outputs/renders/anim/spectrum_wifi.gif)

<sub>WiFi 파형 스펙트럼(넓은 대역) — 시간파형·스펙트럼 모두 Sionna 와 상관 1.0000 로 일치했다.</sub>

In [ ]:
# §3 재현 — 자작 파형과 Sionna 파형을 겹쳐 계산한 상관·NMSE 를 그대로 읽는다 (하드코딩 없음)
import json
J = json.load(open('outputs/report2_waveform_rcs.json'))
for k in ('wifi', 'lte', 'nr'):
    d = J['crosscheck'][k]
    print(f"{d['name']:14s}  표본={d['n']:>7,}  "
          f"상관={d['corr']:.4f}  NMSE={d['nmse_db']:7.1f} dB")

# 대조 자체를 처음부터 다시 돌리려면:  python src/waveforms_sionna.py

---
## §4. 이 대조가 잡아내는 것 — 슬롯 첫 심볼의 긴 CP

> 🔍 **여기서 하는 일** — 상관 1.0000 이 얼마나 예민한 성적표인지, 규격의 깨알 조항 하나로 보여줍니다.

상관이 1.0000 이라는 건 그냥 '대충 비슷하다' 가 아니라 **규격의 미세한 부분까지 전부 맞았다**는 뜻입니다. 그걸 잘 보여주는 조항이 **순환전치(CP)의 길이 규칙** 입니다.

**CP 는 각 OFDM 심볼의 끝부분을 복사해 앞에 덧붙인 완충 구간**입니다(다중경로로 신호가 번져도 심볼끼리 안 겹치게 막는 보호띠). 그런데 3GPP 는 한 슬롯의 **첫 심볼에만 이 완충을 조금 더 길게** 주라고 정해뒀습니다 — 슬롯 경계를 정렬하려는 목적입니다. 측정된 CP 길이를 보면 규칙이 그대로 드러납니다:

| 표준 | 앞쪽 심볼들의 CP 길이 (샘플) | CP 가 균일한가 |
|---|---|---|
| WiFi 802.11ac | [64] | ✅ 균일 (첫 심볼도 같음) |
| LTE Rel-9 | [160, 144, 144, 144, …] | ❌ **첫 칸만 160, 이후 144** |
| 5G NR Rel-16 | [352, 288, 288, 288, …] | ❌ **첫 칸만 352, 이후 288** |

**이 규칙 하나가 얼마나 중요한지**, 대조의 예민함으로 확인할 수 있습니다. 만약 이 '첫 심볼 긴 CP' 를 놓치고 **모든 심볼에 같은 CP** 를 줬다면, 두 번째 심볼부터 시간축이 어긋나 파형 전체가 밀립니다. 그 결과 상관은:

| 표준 | CP 규칙을 지켰을 때 | 첫-심볼 긴 CP 를 놓쳤을 때 |
|---|---|---|
| WiFi 802.11ac (원래 균일) | 1.0000 | 1.0000 *(변화 없음 — CP 가 원래 균일해서)* |
| LTE Rel-9 | 1.0000 | **0.0634** ← 무너짐 |
| 5G NR Rel-16 | 1.0000 | **0.0451** ← 무너짐 |

LTE·5G 는 상관이 1.0000 에서 **0.06·0.05 로 폭락**합니다 — 즉 이 대조는 **CP 배치 규칙 하나까지 잡아낼 만큼 예민**하다는 뜻입니다. (WiFi 는 원래 CP 가 균일해서 이 조항이 없고, 그래서 값이 안 변합니다 — 한 신호만 봐서는 걸러낼 수 없을 종류의 미세한 차이가, **여러 표준을 함께 대조**하기에 드러납니다.)

![crosscheck](outputs/figures/report2_crosscheck.png)

> **정리 —** 상관 1.0000 은 '눈으로 봐서 비슷하다' 와는 차원이 다른 성적표입니다. 슬롯 첫 심볼의 CP 를 한 조각만 틀려도 0.05 로 무너지는 시험을, 세 신호가 전부 **만점**으로 통과했습니다. 우리가 손으로 만든 파형은 규격과 **사실상 완전히 같습니다.**

---
> **다음 리포트**: report06 — 조명(파형)이 규격대로임을 확인했으니, 이제 **표적 쪽** 으로 갑니다. 드론이 레이더 전파를 **얼마나 밝게 되비치는가**(RCS — 레이더 되비침 밝기)를 다룹니다. 표적이 밝아야 잡히니, 이건 탐지의 다른 절반입니다.